# 17 · Advanced Joins

You know `INNER`/`LEFT`/self joins. Engineers also need the full toolbox:
- `CROSS JOIN` (Cartesian product) and when it's useful
- `RIGHT JOIN` and `FULL OUTER JOIN` (SQLite 3.39+)
- `USING` vs `ON`, and the danger of `NATURAL JOIN`
- **non-equi joins** (join on `<`, `>`, `BETWEEN`, not just `=`)
- **semi-joins** (`EXISTS`/`IN`) vs **anti-joins** (`NOT EXISTS`)
- the crucial difference between filtering in `ON` vs `WHERE` for outer joins

In [ ]:
# ▶ Run this cell first. It loads JupySQL and connects to the SQLite database.
%load_ext sql
from sqlalchemy import create_engine
import os

# Works whether the notebook's working dir is the repo root or notebooks/
db_path = 'data/retail.db' if os.path.exists('data/retail.db') else '../data/retail.db'
engine = create_engine(f'sqlite:///{db_path}')

%config SqlMagic.autopandas = True      # results come back as pandas DataFrames
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = 0
%config SqlMagic.displaylimit = 100

%sql engine
print('Connected to', db_path)

## `CROSS JOIN` — every combination
Produces every pairing of left × right rows. Great for building grids/matrices,
e.g. every category across every quarter (even ones with no data):

In [ ]:
%%sql
WITH quarters(quarter) AS (VALUES ('Q1'), ('Q2'), ('Q3'), ('Q4'))
SELECT c.category_name, quarters.quarter
FROM categories c
CROSS JOIN quarters
ORDER BY c.category_name, quarters.quarter
LIMIT 12;

## `USING` vs `ON`
When the join columns have the **same name** in both tables, `USING (col)` is
shorthand for `ON a.col = b.col` and it collapses the duplicate column into one.

In [ ]:
%%sql
SELECT product_name, category_name
FROM products
JOIN categories USING (category_id)
LIMIT 5;

## `NATURAL JOIN` — convenient but risky
`NATURAL JOIN` auto-joins on **all** identically-named columns. It works here
because `products` and `categories` share only `category_id` — but it silently
breaks the day someone adds another same-named column (like `created_at`). Prefer
explicit `ON`/`USING` in real code.

In [ ]:
%%sql
SELECT product_name, category_name FROM products NATURAL JOIN categories LIMIT 5;

## Non-equi join
The `ON` condition doesn't have to be `=`. Here we join each product to every
*more expensive* product to count how many products outrank it on price — a
non-equi self-join.

In [ ]:
%%sql
SELECT p1.product_name, p1.unit_price,
       COUNT(p2.product_id) AS pricier_products
FROM products p1
LEFT JOIN products p2 ON p2.unit_price > p1.unit_price
GROUP BY p1.product_id, p1.product_name, p1.unit_price
ORDER BY pricier_products
LIMIT 8;

## `RIGHT JOIN` and `FULL OUTER JOIN`
`RIGHT JOIN` keeps all rows of the *right* table; `FULL OUTER JOIN` keeps
unmatched rows from **both** sides (NULLs where there's no match). Compare a
target list of countries against the countries we actually have customers in:

In [ ]:
%%sql
WITH target(country) AS (VALUES ('USA'), ('UK'), ('Mars')),
     have AS (SELECT DISTINCT country FROM customers)
SELECT target.country AS target_country,
       have.country   AS customer_country
FROM target
FULL OUTER JOIN have ON target.country = have.country
ORDER BY target_country;

`Mars` appears with a NULL match (a target with no customers); the real customer countries not in the target list appear with a NULL target. That two-sided view is what `FULL OUTER JOIN` is for.

## Semi-join vs anti-join
A **semi-join** returns left rows that *have* a match (without duplicating them) —
express it with `EXISTS` or `IN`. An **anti-join** returns left rows with *no*
match — `NOT EXISTS`.

In [ ]:
%%sql
-- SEMI-JOIN: products that have been ordered
SELECT product_name FROM products p
WHERE EXISTS (SELECT 1 FROM order_items oi WHERE oi.product_id = p.product_id)
ORDER BY product_name;

In [ ]:
%%sql
-- ANTI-JOIN: products never ordered
SELECT product_name FROM products p
WHERE NOT EXISTS (SELECT 1 FROM order_items oi WHERE oi.product_id = p.product_id)
ORDER BY product_name;

## ⚠️ `ON` vs `WHERE` in outer joins — a classic trap
With a `LEFT JOIN`, a condition on the **right** table placed in `WHERE` filters
out the NULL-extended rows, silently turning your `LEFT JOIN` back into an
`INNER JOIN`. Put such conditions in `ON` to keep unmatched left rows.

Condition in `ON` (keeps every customer, only counts *completed* orders):

In [ ]:
%%sql
SELECT cu.first_name, COUNT(o.order_id) AS completed_orders
FROM customers cu
LEFT JOIN orders o ON o.customer_id = cu.customer_id AND o.status = 'completed'
GROUP BY cu.customer_id, cu.first_name
ORDER BY completed_orders;

Same filter in `WHERE` instead — customers with **zero** completed orders vanish, because their single NULL row fails `status = 'completed'`:

In [ ]:
%%sql
SELECT cu.first_name, COUNT(o.order_id) AS completed_orders
FROM customers cu
LEFT JOIN orders o ON o.customer_id = cu.customer_id
WHERE o.status = 'completed'
GROUP BY cu.customer_id, cu.first_name
ORDER BY completed_orders;

## Practice

**✏️ Exercise 1.** Use a FULL OUTER JOIN to list every country that appears as either a customer country or a supplier country, showing which side(s) it came from.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
WITH cust AS (SELECT DISTINCT country FROM customers),
     sup  AS (SELECT DISTINCT country FROM suppliers)
SELECT cust.country AS customer_country, sup.country AS supplier_country
FROM cust FULL OUTER JOIN sup ON cust.country = sup.country
ORDER BY COALESCE(cust.country, sup.country);

**✏️ Exercise 2.** Using an anti-join, list employees who have never been assigned to an order.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
SELECT first_name, last_name FROM employees e
WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.employee_id = e.employee_id);

### ✅ Recap
`CROSS JOIN` builds combinations; `RIGHT`/`FULL OUTER` keep unmatched rows;
`USING` is tidy same-name joining while `NATURAL JOIN` is fragile; non-equi joins
match on ranges; `EXISTS`/`NOT EXISTS` are semi/anti-joins; and outer-join
filters belong in `ON`, not `WHERE`.

**Next:** `18_advanced_aggregation.ipynb`.